In [ ]:
!pip -q install -U peft

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

In [ ]:
OUT_TRAIN = "/content/drive/MyDrive/Elbraka/train.jsonl"
OUT_VALID = "/content/drive/MyDrive/Elbraka/valid.jsonl"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
ds = load_dataset("json", data_files={"train": OUT_TRAIN, "validation": OUT_VALID})

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def apply_template(example):
    msgs = example["messages"]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = ds.map(apply_template, remove_columns=ds["train"].column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:
MAX_LEN = 1024  # لتوفير موارد Colab
def tokenize(batch):
    enc = tok(batch["text"], truncation=True, max_length=MAX_LEN, padding="max_length")
    labels = enc["input_ids"].copy()
    pad = tok.pad_token_id
    enc["labels"] = [[(t if t != pad else -100) for t in seq] for seq in labels]
    return enc

ds_tok = ds.map(tokenize, batched=True)
ds_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules
)

model = get_peft_model(base, lora)
model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


In [ ]:
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Elbraka/lora_out",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=50,
    save_steps=300,
    eval_steps=300,
    eval_strategy="steps",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"]
)

trainer.train()

trainer.save_model("/content/drive/MyDrive/Elbraka/lora_adapter")
tok.save_pretrained("/content/drive/MyDrive/Elbraka/lora_adapter")
print("Saved -> /content/drive/MyDrive/Elbraka/lora_adapter")

Step,Training Loss,Validation Loss
300,0.161468,0.157646
600,0.118149,0.112936
900,0.083534,0.076924
1200,0.058330,0.060169
1500,0.054918,0.053530
1800,0.050791,0.050252


Saved -> /content/drive/MyDrive/Elbraka/lora_adapter
